##**Modulo 1 Extraccion y Almacenamiento de datos**

1 instalamos librerarias para poder utilizar sus funcionalidades que traen

2 importamos librerias para usar en el codigo

3 definimos funciones para utilizar alo largo de la extraccion y almacenamiento de datos

4 empezamos con la extraccion de datos de la api por medio de su url, defino los endpoints, obtenemos los datos de cada endpoint en formato json y lo guardamos en json_latest y json_evolution, confirmo datos y los paso a dataframe y los imprimo para asegurar que los tengo

5 definimos la zona horaria que es la Argentina
en un principio yo queria hacer una medicioion por cada hora de los datos dt_latest, pero sucedio algo inesperado

la api solo actualiza los dias de semana, sabado y domingo no actualiza, me la re bajo pero estoy casi seguro que si actualizamos el programa el lunes y martes nos devolveran datos y los guardara

6 ahi empiezo con la extraccion incremental pero
como la api solo devuelve el ultimo registro anidado, cree una tabla para que guarden los datos de la ultima actualizacion, para que cada vez que haya una actualizacion y la compara con la actual, si es distinto la cambia, y de esa manera hacer una extraccion incremenetal, evito duplicados y guardo si los datos son recientes

7 Hago el full extract a df_evolution que literalmente son registros guardados desde 2011 hasta ahora, dando nos exactamente(hasta ahora) 9005 registros

8 implemento las opciones para vincular la base de datos con MinIO osea el delta lake

9 al guiarme con los date en el dt_latest, hago que cree una columna particion por fecha,

leo la tabla delta existente si no existe la creo, y mantengo solo los nuevos registros, y los guardo solo si son nuevos con el mode append

leo la tabla delta desde minio y convierto a pandas, para mirar las primeras filas

10 aseguro datos y elimino duplicados por date y source, guardo todo el historico en deltalake con modo overwrite y le pido cuantos registros devuelve, exactamente los 9005

defino la ruta de la tabla, leo la tabla de nuevo para validar y mostrar

 # **Extracción de datos de la API Bluelytics**

##**Instalamos e Importamos Librerias**

In [183]:
!pip install requests
!pip install deltalake
!pip install pyarrow

In [184]:
#definimos las librerias utilizadas
import requests
import pandas as pd
import os
import json
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import pyarrow as pa
from deltalake import write_deltalake, DeltaTable
from deltalake.exceptions import TableNotFoundError

In [185]:
def get_data(base_url, endpoint, data_field=None, params=None, headers=None):
    """
    Realiza una solicitud GET a una API para obtener datos.

    Parámetros:
    base_url (str): La URL base de la API.
    endpoint (str): El endpoint de la API al que se realizará la solicitud.
    params (dict): Parámetros de consulta para enviar con la solicitud.
    data_field (str): El nombre del campo en el JSON que contiene los datos.
    headers (dict): Encabezados para enviar con la solicitud.

    Retorna:
    dict: Los datos obtenidos de la API en formato JSON.
    """
    try:
        endpoint_url = f"{base_url}/{endpoint}"
        response = requests.get(endpoint_url, params=params, headers=headers)
        response.raise_for_status()  # Levanta una excepción si hay un error en la respuesta HTTP.

        # Verificar si los datos están en formato JSON.
        try:
            data = response.json()
            if data_field:
              data = data[data_field]
        except:
            print("El formato de respuesta no es el esperado")
            return None
        return data

    except requests.exceptions.RequestException as e:
        # Capturar cualquier error de solicitud, como errores HTTP.
        print(f"La petición ha fallado. Código de error : {e}")
        return None

def build_table(json_data):
    """
    Construye un DataFrame de pandas a partir de datos en formato JSON.

    Parámetros:
    json_data (dict or list(dict)): Los datos en formato JSON obtenidos de una API.

    Retorna:
    DataFrame: Un DataFrame de pandas que contiene los datos.
    """
    try:
        df = pd.json_normalize(
            json_data
            )
        return df
    except:
        print("Los datos no están en el formato esperado")
        return None

In [186]:
def save_data_as_delta(df, path, storage_options, mode="overwrite", partition_cols=None):
    """
    Guarda un dataframe en formato Delta Lake en la ruta especificada.
    A su vez, es capaz de particionar el dataframe por una o varias columnas.
    Por defecto, el modo de guardado es "overwrite".

    Args:
      df (pd.DataFrame): El dataframe a guardar.
      path (str): La ruta donde se guardará el dataframe en formato Delta Lake.
      mode (str): El modo de guardado. Son los modos que soporta la libreria
      deltalake: "overwrite", "append", "error", "ignore".
      partition_cols (list or str): La/s columna/s por las que se particionará el
      dataframe. Si no se especifica, no se particionará.
    """
    write_deltalake(
        path, df, mode=mode, storage_options=storage_options, partition_by=partition_cols
    )

def save_new_data_as_delta(new_data, data_path, predicate, storage_options, partition_cols=None):
    """
    Guarda solo nuevos datos en formato Delta Lake usando la operación MERGE,
    comparando los datos ya cargados con los datos que se desean almacenar
    asegurando que no se guarden registros duplicados.

    Args:
      new_data (pd.DataFrame): Los datos que se desean guardar.
      data_path (str): La ruta donde se guardará el dataframe en formato Delta Lake.
      predicate (str): La condición de predicado para la operación MERGE.
    """

    try:
      dt = DeltaTable(data_path, storage_options=storage_options)
      new_data_pa = pa.Table.from_pandas(new_data)
      # Se insertan en target, datos de source que no existen en target
      dt.merge(
          source=new_data_pa,
          source_alias="source",
          target_alias="target",
          predicate=predicate
      ) \
      .when_not_matched_insert_all() \
      .execute()

    # Si no existe la tabla Delta Lake, se guarda como nueva
    except TableNotFoundError:
      save_data_as_delta(new_data, data_path, storage_options=storage_options, partition_cols=partition_cols)

def upsert_data_as_delta(data, data_path, predicate, storage_options):
    """
    Guardar datos en formato Delta Lake usando la operacion MERGE.
    Cuando no haya registros coincidentes, se insertarán nuevos registros.
    Cuando haya registros coincidentes, se actualizarán los campos.

    Args:
      data (pd.DataFrame): Los datos que se desean guardar.
      data_path (str): La ruta donde se guardará el dataframe en formato Delta Lake.
      predicate (str): La condición de predicado para la operación MERGE.
    """
    try:
        dt = DeltaTable(data_path)
        data_pa = pa.Table.from_pandas(data)
        dt.merge(
            source=data_pa,
            source_alias="source",
            target_alias="target",
            predicate=predicate
        ) \
        .when_matched_update_all() \
        .when_not_matched_insert_all() \
        .execute()
    except TableNotFoundError:
        save_data_as_delta(data, data_path, storage_options)

##**Extraemos de la API los Dataframe en formato JSON**

In [187]:
#paso la url raiz de la api que voy a utilizar
base_url = "https://api.bluelytics.com.ar"

In [188]:
#defino los endpoints
endpoint_latest = "v2/latest"
endpoint_evolution = "v2/evolution.json"

In [189]:
#obtenemos y guardamos la respuesta de json de cada endpoint uando get data
json_latest = get_data(
    base_url, endpoint_latest,
    params=None, data_field=None
    )

json_evolution = get_data(
    base_url, endpoint_evolution,
    params=None, data_field=None
    )

In [190]:
#confirmo que tengo los datos solicitados
print(json_latest)
print(json_evolution)

{'oficial': {'value_avg': 1374.0, 'value_sell': 1399.0, 'value_buy': 1349.0}, 'blue': {'value_avg': 1380.0, 'value_sell': 1390.0, 'value_buy': 1370.0}, 'oficial_euro': {'value_avg': 1493.5, 'value_sell': 1521.0, 'value_buy': 1466.0}, 'blue_euro': {'value_avg': 1500.0, 'value_sell': 1511.0, 'value_buy': 1489.0}, 'last_update': '2026-04-10T19:45:56.551625-03:00'}
[{'date': '2026-04-10', 'source': 'Oficial', 'value_sell': 1400.0, 'value_buy': 1350.0}, {'date': '2026-04-10', 'source': 'Blue', 'value_sell': 1390.0, 'value_buy': 1370.0}, {'date': '2026-04-09', 'source': 'Oficial', 'value_sell': 1410.0, 'value_buy': 1360.0}, {'date': '2026-04-09', 'source': 'Blue', 'value_sell': 1390.0, 'value_buy': 1370.0}, {'date': '2026-04-08', 'source': 'Oficial', 'value_sell': 1412.0, 'value_buy': 1362.0}, {'date': '2026-04-08', 'source': 'Blue', 'value_sell': 1390.0, 'value_buy': 1370.0}, {'date': '2026-04-07', 'source': 'Oficial', 'value_sell': 1419.0, 'value_buy': 1369.0}, {'date': '2026-04-07', 'sour

In [191]:
#convierto el json a dataframr
df_latest = build_table(json_latest)

#imprimo los resultados de comprar y venta de monedas
df_latest.head()

,last_update,oficial.value_avg,oficial.value_sell,oficial.value_buy,blue.value_avg,blue.value_sell,blue.value_buy,oficial_euro.value_avg,oficial_euro.value_sell,oficial_euro.value_buy,blue_euro.value_avg,blue_euro.value_sell,blue_euro.value_buy
0,2026-04-10T19:45:56.551625-03:00,1374.0,1399.0,1349.0,1380.0,1390.0,1370.0,1493.5,1521.0,1466.0,1500.0,1511.0,1489.0


In [192]:
#convierto el json en dataframe
df_evolution = build_table(json_evolution)

#imprimo resultados de compra/venta de dolar oficial/blue, con datos historicos
df_evolution
#o df_evolution que mostrara los primeros 5 y ultimos 5 registros
#de 2011 a 2026

,date,source,value_sell,value_buy
0,2026-04-10,Oficial,1400.0,1350.0
1,2026-04-10,Blue,1390.0,1370.0
2,2026-04-09,Oficial,1410.0,1360.0
3,2026-04-09,Blue,1390.0,1370.0
4,2026-04-08,Oficial,1412.0,1362.0
...,...,...,...,...
9040,2011-01-05,Blue,4.0,4.0
9041,2011-01-04,Oficial,4.0,4.0
9042,2011-01-04,Blue,4.0,4.0
9043,2011-01-03,Oficial,4.0,4.0


In [193]:
#valido y limpio algunos datos
df_latest = df_latest.drop_duplicates()
df_latest.columns = df_latest.columns.str.lower()

df_evolution = df_evolution.drop_duplicates()
df_evolution.columns = df_evolution.columns.str.lower()

## **Obtener el valor de la ultima hora de cada compra y venta**

In [194]:
#defino las varibales que tomare de tiempo, no utilizo UTC porq es horario arg
tz = ZoneInfo("America/Argentina/Buenos_Aires")
#hora actual
now_local = datetime.now(tz)

#obtengo la ultima hora,redondeada a la actual, hace 1 y 2 hrs
current_hour = now_local.replace(minute=0, second=0, microsecond=0)

#convierto a iso
iso_current = current_hour.isoformat()

#imprimo para confirmar
print(iso_current)

2026-04-10T23:00:00-03:00


##**Extraccion incremental**

In [195]:
#voy a crear un guardado por cada vez que se actualize el df_latest
#ya que solo devuelve el ultimo dato
HISTORIC_FILE = "historic_latest.json"
df_latest["hour"] = current_hour.isoformat()

#leemos el historico que ya existe o creo uno nuevo
if os.path.exists(HISTORIC_FILE):
    df_historic = pd.read_json(HISTORIC_FILE)
else:
    df_historic = pd.DataFrame()

#evito duplicado, solo añadir si la hora ya paso
if not df_historic.empty and current_hour.isoformat() in df_historic.get("hour", []):
    print("ya existe un registro para esta hora. No se añade. ")
else:
    df_historic = pd.concat([df_historic, df_latest], ignore_index=True)
    #guardo historico en json
    df_historic.to_json(HISTORIC_FILE, orient="records", date_format="iso")
    print(f"Historico actualizado. Registro añadido para{current_hour.isoformat()}")

#lo muestra epicamente
df_latest


Historico actualizado. Registro añadido para2026-04-10T23:00:00-03:00


,last_update,oficial.value_avg,oficial.value_sell,oficial.value_buy,blue.value_avg,blue.value_sell,blue.value_buy,oficial_euro.value_avg,oficial_euro.value_sell,oficial_euro.value_buy,blue_euro.value_avg,blue_euro.value_sell,blue_euro.value_buy,hour
0,2026-04-10T19:45:56.551625-03:00,1374.0,1399.0,1349.0,1380.0,1390.0,1370.0,1493.5,1521.0,1466.0,1500.0,1511.0,1489.0,2026-04-10T23:00:00-03:00


In [196]:
# último timestamp guardado en tu histórico
if not df_historic.empty:
    last_saved = pd.to_datetime(df_historic["last_update"]).max()
else:
    last_saved = None

# timestamp del registro actual
current_update = pd.to_datetime(df_latest["last_update"].iloc[0])

# solo guardar si es más reciente
if last_saved is None or current_update > last_saved:
    df_historic = pd.concat([df_historic, df_latest], ignore_index=True)
    df_historic.to_json(HISTORIC_FILE, orient="records", date_format="iso", indent=4)
    print(f"Nuevo registro guardado: {current_update}")
else:
    print("No hay registro nuevo. No se guarda.")

No hay registro nuevo. No se guarda.


In [197]:
#verifico que las tablas esten god
with open("historic_latest.json", "r") as f:
    data = json.load(f)

# Mostrar los primeros 3 registros de forma legible
print(json.dumps(data[:2], indent=4))

[
    {
        "last_update": "2026-04-10T19:45:56.551625-03:00",
        "oficial.value_avg": 1374.0,
        "oficial.value_sell": 1399.0,
        "oficial.value_buy": 1349.0,
        "blue.value_avg": 1380.0,
        "blue.value_sell": 1390.0,
        "blue.value_buy": 1370.0,
        "oficial_euro.value_avg": 1493.5,
        "oficial_euro.value_sell": 1521.0,
        "oficial_euro.value_buy": 1466.0,
        "blue_euro.value_avg": 1500.0,
        "blue_euro.value_sell": 1511.0,
        "blue_euro.value_buy": 1489.0,
        "hour": "2026-04-10T20:00:00-03:00"
    },
    {
        "last_update": "2026-04-10T19:45:56.551625-03:00",
        "oficial.value_avg": 1374.0,
        "oficial.value_sell": 1399.0,
        "oficial.value_buy": 1349.0,
        "blue.value_avg": 1380.0,
        "blue.value_sell": 1390.0,
        "blue.value_buy": 1370.0,
        "oficial_euro.value_avg": 1493.5,
        "oficial_euro.value_sell": 1521.0,
        "oficial_euro.value_buy": 1466.0,
        "blue_e

## **Extracción FULL**

In [198]:
#archivo donde guardo el historico completo
HISTORIC_FILE = "historic_full.json"

#df_evolucion: dataframe que obtuve de la api
#convierto la columna date a date time
df_evolution['date'] = pd.to_datetime(df_evolution['date'])

#elimino duplicado
df_evolution = df_evolution.drop_duplicates(subset=['date', 'source'])

#Guardo todo historico completo con todo lo que tiene en json
df_evolution.to_json(HISTORIC_FILE,
                     orient="records", #cada fila como dict
                     date_format="iso", #fechas en iso
                     indent=4 #formato ligible
    )

#imprimo el total de registros
print(f"FULL extract completo en formato JSON. Total de registros: {len(df_evolution)} ")

FULL extract completo en formato JSON. Total de registros: 9045 


In [199]:
#verifico y observo que las tablas esten god
# Abro el JSON del full extract
with open("historic_full.json", "r") as f:
    data = json.load(f)

# Mostrar los primeros 3 registros de forma legible
print(json.dumps(data[:3], indent=4))

[
    {
        "date": "2026-04-10T00:00:00.000",
        "source": "Oficial",
        "value_sell": 1400.0,
        "value_buy": 1350.0
    },
    {
        "date": "2026-04-10T00:00:00.000",
        "source": "Blue",
        "value_sell": 1390.0,
        "value_buy": 1370.0
    },
    {
        "date": "2026-04-09T00:00:00.000",
        "source": "Oficial",
        "value_sell": 1410.0,
        "value_buy": 1360.0
    }
]


## **Almacenamiento de datos en MinIO**

In [200]:
# Configuraciones para MinIO
storage_options = {
    'AWS_ENDPOINT_URL': 'http://31.97.241.212:9002',
    'AWS_ACCESS_KEY_ID': 'claudioquispe', # username
    'AWS_SECRET_ACCESS_KEY': 'claudioquispe', # contraseña
    'AWS_ALLOW_HTTP': 'true',
    'aws_conditional_put': 'etag',
    'AWS_S3_ALLOW_UNSAFE_RENAME': 'true'
}

bkt_name = "claudioquispe-bucket"

##**Latest_append**

In [201]:
#convierto los timestamps y creo una columma de particion

df_latest["last_update"] = pd.to_datetime(df_latest["last_update"], utc=True)

df_latest["hour"] = df_latest["last_update"].dt.floor("H")

#particion por fecha
df_latest["date"] = df_latest["last_update"].dt.floor("D")

/tmp/ipykernel_1584/739411334.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_latest["hour"] = df_latest["last_update"].dt.floor("H")


In [202]:
# Leer la tabla Delta existente
try:
    dt = DeltaTable(f"s3://{bkt_name}/data/bronze/latest_append", storage_options=storage_options)
    df_existing = dt.to_pandas()
except:
    df_existing = pd.DataFrame()

# Mantener solo los nuevos registros
if not df_existing.empty:
    df_to_write = df_latest[~df_latest["last_update"].isin(df_existing["last_update"])]
else:
    df_to_write = df_latest

# Guardar solo si hay nuevos registros
if not df_to_write.empty:
    write_deltalake(
        f"s3://{bkt_name}/data/bronze/latest_append",
        df_to_write,
        mode="append",
        storage_options=storage_options,
        schema_mode="merge"
    )

In [203]:
# Leer la tabla Delta desde MinIO y convertir a pandas
df_latest_from_delta = DeltaTable(
    f"s3://{bkt_name}/data/bronze/latest_append",
    storage_options=storage_options
).to_pandas()

# Mostrar las primeras filas
df_latest_from_delta.head()

,last_update,oficial.value_avg,oficial.value_sell,oficial.value_buy,blue.value_avg,blue.value_sell,blue.value_buy,oficial_euro.value_avg,oficial_euro.value_sell,oficial_euro.value_buy,blue_euro.value_avg,blue_euro.value_sell,blue_euro.value_buy,hour,date
0,2026-04-10 22:45:56.551625+00:00,1374.0,1399.0,1349.0,1380.0,1390.0,1370.0,1493.5,1521.0,1466.0,1500.0,1511.0,1489.0,2026-04-10 22:00:00+00:00,2026-04-10
1,2026-04-10 21:01:00.781542+00:00,1374.0,1399.0,1349.0,1380.0,1390.0,1370.0,1493.5,1521.0,1466.0,1500.0,1511.0,1489.0,2026-04-10 21:00:00+00:00,2026-04-10
2,2026-04-09 19:46:05.770421+00:00,1384.5,1410.0,1359.0,1380.0,1390.0,1370.0,1505.5,1533.0,1478.0,1500.0,1511.0,1489.0,2026-04-09 19:00:00+00:00,2026-04-09
3,2026-04-09 17:15:58.754426+00:00,1385.0,1410.0,1360.0,1380.0,1390.0,1370.0,1505.5,1533.0,1478.0,1500.0,1511.0,1489.0,2026-04-09 17:00:00+00:00,2026-04-09
4,2026-03-27 22:45:56.776664+00:00,1379.0,1404.0,1354.0,1405.0,1415.0,1395.0,1499.0,1527.0,1471.0,1527.0,1538.0,1516.0,2026-03-28 17:00:00+00:00,2026-03-27


##**Evolution_full**

In [204]:
# Asegurarse que la columna date sea datetime
df_evolution['date'] = pd.to_datetime(df_evolution['date'])

# Eliminar duplicados por 'date' y 'source'
df_evolution = df_evolution.drop_duplicates(subset=['date', 'source'])

# Definir la ruta de la tabla Delta
delta_path = f"s3://{bkt_name}/data/bronze/evolution_full"

# Guardar todo el histórico en Delta Lake (modo overwrite)
write_deltalake(
    delta_path,
    df_evolution,
    mode="overwrite", # Sobrescribe la tabla completa
    storage_options=storage_options,
    schema_mode="merge"
)

print(f" La Tabla de full extract encuentra esta cantidad de registros: {len(df_evolution)}")

 La Tabla de full extract encuentra esta cantidad de registros: 9045


In [205]:
# Leer la tabla de vuelta para validar
df_from_delta = DeltaTable(
    delta_path,
    storage_options=storage_options
).to_pandas()

# Mostrar primeras filas
df_from_delta.head()

,date,source,value_sell,value_buy,__index_level_0__
0,2026-04-10,Oficial,1400.0,1350.0,NaN
1,2026-04-10,Blue,1390.0,1370.0,NaN
2,2026-04-09,Oficial,1410.0,1360.0,NaN
3,2026-04-09,Blue,1390.0,1370.0,NaN
4,2026-04-08,Oficial,1412.0,1362.0,NaN


#**INICIO DEL TP 2**

##**Eliminacion de Nulos y duplicados**

In [206]:
# Leo Delta
dt = DeltaTable(delta_path, storage_options=storage_options)
df = dt.to_pandas()

# 1. Limpieza de datos
# Limpio índice extra
df = df.reset_index(drop=True)

# elimino index si existe
if "__index_level_0__" in df.columns:
    df = df.drop(columns=["__index_level_0__"])

# 2. conversion de tipos, convierto a fecha
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# 3. Elimino nulos
df = df.dropna(subset=['date', 'source', 'value_sell', 'value_buy'])

# 4. Ordenamiento para definir que registro conservo
df = df.sort_values(by="date", ascending=False)

# 5.Elimino duplicados(con criterio)
df = df.drop_duplicates(subset=['date', 'source'], keep="first")

print(f"Registros finales limpios: {len(df)}")

Registros finales limpios: 9045


##**Tipos de Datos**


In [207]:
# Convierto a tipos
df = df.astype({
    "source": "category",
    "value_sell": "float32",
    "value_buy": "float32"
})

In [208]:
silver_path = f"s3://{bkt_name}/data/silver/evolution_clean"

# lo guardo en silver
save_new_data_as_delta(
    df,
    silver_path,
    "target.date = source.date AND target.source = source.source",
    storage_options=storage_options
)

##**Renombrar columnas, formatear columnas de fecha**

In [209]:
# Creo un diccionario de renombrado
new_columns = {col: col.replace("oficial", "f").replace("blue", "b")
               for col in df_latest.columns}
# Renombro las columnas
df_latest = df_latest.rename(columns=new_columns)

# Reviso resultado
df_latest.head()

,last_update,f.value_avg,f.value_sell,f.value_buy,b.value_avg,b.value_sell,b.value_buy,f_euro.value_avg,f_euro.value_sell,f_euro.value_buy,b_euro.value_avg,b_euro.value_sell,b_euro.value_buy,hour,date
0,2026-04-10 22:45:56.551625+00:00,1374.0,1399.0,1349.0,1380.0,1390.0,1370.0,1493.5,1521.0,1466.0,1500.0,1511.0,1489.0,2026-04-10 22:00:00+00:00,2026-04-10 00:00:00+00:00


In [210]:
# convierto a timestamp
df_latest["last_update"] = pd.to_datetime(df_latest["last_update"], utc=True)
#quito zona horaria si delta lake lo requiere
df_latest["last_update"] = df_latest["last_update"].dt.tz_localize(None)

# Extraigo columnas fecha y hora
df_latest["date"] = df_latest["last_update"].dt.floor("D") # solo fecha
df_latest["hour"] = df_latest["last_update"].dt.hour  # solo hora

# convierto otro tipo de datos si hace falta
type_mapping = {
    "f.value_avg": "float32",
    "f.value_sell": "float32",
    "f.value_buy": "float32",
    "b.value_avg": "float32",
    "b.value_sell": "float32",
    "b.value_buy": "float32",
    "date": "datetime64[ns]",
    "hour": "string"
}

for col, typ in type_mapping.items():
    if col in df_latest.columns:
        df_latest[col] = df_latest[col].astype(typ)

In [211]:
# limpieza del latest para pasarlo al limpio al silver
df_latest = df_latest.dropna(subset=["date"])

#ordeno para poder limpiar de forma eficiente
df_latest = df_latest.sort_values(by="last_update", ascending=False)

# elimino duplicados
df_latest = df_latest.drop_duplicates(subset=["date", "hour"], keep="first")

# limpio índice
df_latest = df_latest.reset_index(drop=True)

In [212]:
#vistazo rapido de como quedo la tabla
df_latest.head()

,last_update,f.value_avg,f.value_sell,f.value_buy,b.value_avg,b.value_sell,b.value_buy,f_euro.value_avg,f_euro.value_sell,f_euro.value_buy,b_euro.value_avg,b_euro.value_sell,b_euro.value_buy,hour,date
0,2026-04-10 22:45:56.551625,1374.0,1399.0,1349.0,1380.0,1390.0,1370.0,1493.5,1521.0,1466.0,1500.0,1511.0,1489.0,22,2026-04-10


In [213]:
la_silver_path = f"s3://{bkt_name}/data/silver/latest_clean"

#Cambio el MERGE para guardarlo al silver limpio
save_new_data_as_delta(
    df_latest,
    la_silver_path,
   "target.date = source.date AND target.hour = source.hour",
    storage_options=storage_options
)

##**Funciones de Min, MAx y el AVG de Evolution**

In [214]:
#leo desde silver para no reescriber sobre los datos crudos
dt = DeltaTable(silver_path, storage_options=storage_options)
df_silver = dt.to_pandas()

#filtro por año, osea creo una metrica global para el año 2026
df_2026 = df_silver[df_silver["date"].dt.year == 2026]

#creo las agregaciones (min, max, avg)
df_2026 = df_2026.agg({
    "value_sell": ["min", "max", "mean"],
    "value_buy": ["min", "max", "mean"]
})

#formato final para pasarlo a tabla
df_2026 = df_2026.reset_index().rename(columns={"index": "metric"})

df_2026

,metric,value_sell,value_buy
0,min,1390.000000,1344.000000
1,max,1530.000000,1510.000000
2,mean,1442.652832,1407.847168


In [215]:
gold_path = f"s3://{bkt_name}/data/gold/metrics_2026"

# lo guardo en GOLD
write_deltalake(
    gold_path,
    df_2026,
    mode="overwrite",
    storage_options=storage_options
)

##**Datos para su posterior uso**



###**Brecha Cambiaria**

In [216]:
#Brecha cambiaria
df_gap = df_silver.pivot(index="date", columns="source", values="value_sell").reset_index()

df_gap["gap"] = df_gap["Blue"] - df_gap["Oficial"]
df_gap["gap_pct"] = (df_gap["gap"] / df_gap["Oficial"]) * 100

#ordeno para ver desde los ultimos valores
df_gap = df_gap.sort_values("date", ascending=False)

In [217]:
#visualizacion de la esquima
df_gap.head()

source,date,Blue,Oficial,gap,gap_pct
4522,2026-04-10,1390.0,1400.0,-10.0,-0.714286
4521,2026-04-09,1390.0,1410.0,-20.0,-1.418440
4520,2026-04-08,1390.0,1412.0,-22.0,-1.558074
4519,2026-04-07,1395.0,1419.0,-24.0,-1.691332
4518,2026-04-06,1400.0,1417.0,-17.0,-1.199718


In [218]:
#guardo los datos obtenidos en GOLD
gold_gap_path = f"s3://{bkt_name}/data/gold/gap_analysis"

write_deltalake(
    gold_gap_path,
    df_gap,
    mode="overwrite",
    storage_options=storage_options
)

###**Evolucion diaria(variacion del dia a dia)**

In [219]:
#partimos desde el silver
dt = DeltaTable(silver_path, storage_options=storage_options)
df_silver = dt.to_pandas()

#los ordeno de maner clara
df_silver = df_silver.sort_values(by=["source", "date"])

In [220]:
if "__index_level_0__" in df_trend.columns:
    df_trend = df_trend.drop(columns=["__index_level_0__"])

#Evolucion diaria(variacion dia a dia)
df_trend = df_trend.sort_values(by=["source", "date"], ascending=[True, False])

df_trend["daily_change"] = df_trend.groupby("source")["value_sell"].diff()
df_trend["daily_pct"] = df_trend.groupby("source")["value_sell"].pct_change() * 100

#eliminamos el primer valor porq no tiene comparacion
df_trend = df_trend.dropna(subset=["daily_change"])

In [221]:
df_trend.groupby("source").head()

,date,source,value_sell,value_buy,daily_change,daily_pct
9,2026-04-06,Blue,1400.0,1380.0,5.0,0.358427
11,2026-04-03,Blue,1405.0,1385.0,5.0,0.357139
13,2026-04-02,Blue,1405.0,1385.0,0.0,0.000000
15,2026-04-01,Blue,1405.0,1385.0,0.0,0.000000
17,2026-03-31,Blue,1410.0,1390.0,5.0,0.355875
8,2026-04-06,Oficial,1417.0,1365.0,-2.0,-0.140947
10,2026-04-03,Oficial,1416.0,1365.0,-1.0,-0.070572
12,2026-04-02,Oficial,1416.0,1365.0,0.0,0.000000
14,2026-04-01,Oficial,1409.0,1358.0,-7.0,-0.494349
16,2026-03-31,Oficial,1419.0,1369.0,10.0,0.709724


In [222]:
#guardo los datos obtenidos en GOLD
gold_trend_path = f"s3://{bkt_name}/data/gold/daily_trend"

write_deltalake(
    gold_trend_path,
    df_trend,
    mode="overwrite",
    storage_options=storage_options
)

###**Maximo y Minimo historico por año**

In [223]:
#Parto desde silver
dt = DeltaTable(silver_path, storage_options=storage_options)
df_silver = dt.to_pandas()

# 1. creo la columna año
df_silver["year"] = df_silver["date"].dt.year

In [224]:
# 2. si index sigue existiendo lo elimino
if "__index_level_0__" in df_yearly.columns:
    df_yearly = df_yearly.drop(columns=["__index_level_0__"])

# 3. calculo el Maximo y minimo historico por año
df_yearly = df_silver.copy()
df_yearly["year"] = df_yearly["date"].dt.year

df_yearly = df_yearly.groupby(["year", "source"]).agg({
    "value_sell": ["min", "max"]
}).reset_index()

# 4.aplano columnas
df_yearly.columns = ['_'.join(col).strip('_') for col in df_yearly.columns]

#5. visualizo
df_yearly.sort_values(by=["year", "source"], ascending=[False, True]).head(10)

,year,source,value_sell_min,value_sell_max
30,2026,Blue,1390.0,1530.0
31,2026,Oficial,1394.0,1492.0
28,2025,Blue,1150.0,1550.0
29,2025,Oficial,1060.0,1521.0
26,2024,Blue,985.0,1500.0
27,2024,Oficial,847.0,1060.0
24,2023,Blue,346.0,1100.0
25,2023,Oficial,185.0,847.0
22,2022,Blue,195.0,357.0
23,2022,Oficial,108.0,185.0


In [225]:
#guardo los datos obtenidos en GOLD
gold_yearly_path = f"s3://{bkt_name}/data/gold/yearly_min_max"

write_deltalake(
    gold_yearly_path,
    df_yearly,
    mode="overwrite",
    storage_options=storage_options
)